In [ ]:
# ----------------------------- Setup ----------------------------- #
%matplotlib tk
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.widgets import Button
import re

prec = 500
RHP = RealField(prec)
CHP = ComplexField(prec)

# ----------------------------- Input ----------------------------- #
def get_polynomial_coeffs_fixed():
    """
    Robust input for polynomial coefficients:
    - Accepts integers, decimals, scientific notation
    - Strips leading zeros
    - Rejects all-zero polynomials
    """
    pattern = re.compile(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$')
    
    while True:
        s = input("Enter polynomial coefficients (highest degree first): ").strip()
        if not s:
            print("Error: No input provided.")
            continue

        tokens = s.split()
        coeffs = []
        for t in tokens:
            t_clean = t.lstrip("0") or "0"
            if t_clean.startswith("."):
                t_clean = "0" + t_clean
            if not pattern.fullmatch(t_clean):
                print(f"Invalid number: '{t}'")
                break
            try:
                coeffs.append(RHP(t_clean))
            except Exception:
                print(f"Could not convert '{t}' to high-precision number.")
                break
        else:
            # Reject all-zero polynomial
            if all(c == 0 for c in coeffs):
                print("Polynomial cannot be identically zero.")
                continue
            # Strip leading zeros to get true degree
            while coeffs and coeffs[0] == 0:
                coeffs.pop(0)
            return coeffs

coeffs_hp = get_polynomial_coeffs_fixed()

# ----------------------------- Polynomial Construction ----------------------------- #
R.<x> = RHP[]
f = sum(c * x^(len(coeffs_hp) - i - 1) for i, c in enumerate(coeffs_hp))
fC = f.change_ring(CHP)

# ----------------------------- Clean Printing ----------------------------- #
def format_coeff(c, digits=12, tol=1e-30):
    if abs(c) < tol:
        return None
    c = c.n(digits=digits)
    if c.is_integer():
        return str(Integer(c))
    s = f"{c:.12g}"
    if "e" in s:
        base, exp = s.split("e")
        exp = int(exp)
        return f"{base}*10^{exp}"
    return s

def polynomial_to_string(coeffs, var="x", digits=12):
    terms = []
    n = len(coeffs)
    for i, c in enumerate(coeffs):
        power = n - i - 1
        fc = format_coeff(c, digits)
        if fc is None:
            continue
        sign = "-" if fc.startswith("-") else "+"
        fc_abs = fc[1:] if fc.startswith("-") else fc
        if power == 0:
            term = f"{fc_abs}"
        elif power == 1:
            term = var if fc_abs == "1" else f"{fc_abs}*{var}"
        else:
            term = f"{var}^{power}" if fc_abs == "1" else f"{fc_abs}*{var}^{power}"
        terms.append((sign, term))
    if not terms:
        return "0"
    first_sign, first_term = terms[0]
    result = first_term if first_sign == "+" else f"-{first_term}"
    for sign, term in terms[1:]:
        result += f" {sign} {term}"
    return result

poly_str = polynomial_to_string(coeffs_hp)
print(f"\nPolynomial: f(x) = {poly_str} = 0")

# ----------------------------- Roots ----------------------------- #
# For degree 0 constant polynomials, roots list will be empty
roots = fC.roots(multiplicities=True) if len(coeffs_hp) > 1 else []

# ----------------------------- Helpers ----------------------------- #
def poly_eval(coeffs, r):
    p = CHP(0)
    for c in coeffs:
        p = p * r + c
    return p

def relative_residual(coeffs, r):
    numerator = abs(poly_eval(coeffs, r))
    denom = sum(abs(c) * abs(r)^(len(coeffs) - i - 1) for i, c in enumerate(coeffs))
    return numerator / denom if denom != 0 else numerator

# ----------------------------- Print Roots ----------------------------- #
print(f"\nPolynomial degree: {len(coeffs_hp)-1}")
if roots:
    print("Roots with high precision, multiplicity and relative residuals:")
    for i, (r, mult) in enumerate(roots, 1):
        res = relative_residual(coeffs_hp, r)
        print(f"Root {i}: {r.n(prec)} (multiplicity {mult})   Residual: {res:.2e}")
else:
    print("No roots (constant polynomial)")

# ----------------------------- Plotting ----------------------------- #
if roots:
    roots_np = np.array([complex(r) for r, _ in roots])
    multiplicities = [mult for _, mult in roots]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

    # ---- Complex Plane (unchanged) ----
    ax1.axhline(0, color='gray', lw=1)
    ax1.axvline(0, color='gray', lw=1)
    for r, mult in zip(roots_np, multiplicities):
        ax1.scatter(r.real, r.imag, color='red', s=10*mult)
    ax1.set_title("Roots in Complex Plane (size = multiplicity)")
    ax1.set_xlabel("Re")
    ax1.set_ylabel("Im")
    ax1.grid(True)
    ax1.set_aspect('auto')

    # ---- Real Line Plot (improved) ----
    # Identify real roots (same as before)
    real_roots = [(r, mult) for (r, mult) in roots if abs(r.imag()) < RHP('1e-12')]

    # NEW: tighter root-based range + explicit root insertion
    if roots_np.size > 0:
        xmin = float(min(roots_np.real)) - 1.0
        xmax = float(max(roots_np.real)) + 1.0
    else:
        xmin, xmax = -5.0, 5.0

    x_vals_initial = np.linspace(xmin, xmax, 2000)
    x_list = x_vals_initial.tolist()

    # Insert exact real roots so the line always hits y=0 at the markers
    for r, _ in real_roots:
        root_float = float(r.real())
        if not any(abs(root_float - x) < 1e-12 for x in x_list):
            x_list.append(root_float)
    x_list.sort()
    x_vals = np.array(x_list)

    # Evaluate at high precision, then convert to float
    y_vals = np.array([float(f(RHP(xx))) for xx in x_vals])

    # NEW: Clip extreme values — this is what fixes the detached symlog curve
    max_abs = np.max(np.abs(y_vals)) if len(y_vals) > 0 else 1.0
    if np.isinf(max_abs) or max_abs > 1e300:
        print("NOTICE: Polynomial values exceed float range → clipping plot")
        y_vals = np.clip(y_vals, -1e300, 1e300)
        max_abs = 1e300

    ax2.plot(x_vals, y_vals, lw=1, label="f(x)")
    ax2.axhline(0, color='gray', lw=1)

    for r, mult in real_roots:
        ax2.scatter(float(r.real()), 0.0, color='red', s=10*mult, zorder=5)

    ax2.set_title("Polynomial on Real Line (size = multiplicity)")
    ax2.set_xlabel("x")
    ax2.set_ylabel("f(x)")
    ax2.grid(True)
    ax2.legend()

    # Store exact initial limits (used by Linear button)
    init_ylim = ax2.get_ylim()
    
    # ----------------------------- Buttons -----------------------------
    linthresh = 1.0
    
    def set_linear(event):
        ax2.set_yscale('linear')
        ax2.set_ylim(init_ylim)          # use stored limits, not recomputed max_abs
        fig.canvas.draw_idle()
    
    def set_symlog(event):
        ax2.set_yscale('symlog', linthresh=linthresh, linscale=1.0)
        fig.canvas.draw_idle()
    
    axlinear = plt.axes([0.8, 0.02, 0.1, 0.04])
    axsymlog = plt.axes([0.65, 0.02, 0.1, 0.04])
    b_linear = Button(axlinear, 'Linear')
    b_symlog = Button(axsymlog, 'Symlog')
    b_linear.on_clicked(set_linear)
    b_symlog.on_clicked(set_symlog)

    plt.tight_layout()
    plt.show()
else:
    print("No plot: polynomial is constant, no roots to display.")